# MTPC — Full training on Google Colab (Phase 0 → 1 → 2)

Re-trains the byT5-small MTPC models **from scratch** on a GPU runtime, then runs a sanity check.

### Trains WITHOUT `cheat` (on purpose)
`cheat` feeds the whole answer into byT5's **bidirectional** encoder; the decoder then **copies**
the answer instead of predicting it — low training loss but **byte-salad at inference**. Standard
SFT (cheat off) masks the answer in the encoder, so the model learns to predict.

Run cells **1 → 8** in order. First: *Runtime ▸ Change runtime type ▸ GPU* (A100 ideal).

## 1. Mount Drive & clone the repo

In [ ]:
import os, subprocess
print(subprocess.run(["nvidia-smi","-L"], capture_output=True, text=True).stdout
      or "!! No GPU — set Runtime > Change runtime type > GPU, then re-run.")
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/MTPC'          # change if you like
REPO_URL    = 'https://github.com/lorenzoAllegrini/MTPC.git'
if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} "{PROJECT_DIR}"
else:
    !cd "{PROJECT_DIR}" && git fetch origin && git reset --hard origin/main   # pull latest code
%cd {PROJECT_DIR}

## 2. Install dependencies

In [ ]:
!pip install -q -U transformers peft datasets accelerate
# Colab ships an old torchao (0.10.x) that newer PEFT rejects during LoRA creation
# (ImportError: incompatible torchao). We don't use it, so remove it.
!pip uninstall -q -y torchao

## 3. Configuration
Defaults below target an **A100**. `--amp` (bf16 autocast) gives a large speedup on Ampere+ and
halves activation memory. If you hit CUDA OOM on the CP/HMM cells, lower `BATCH_SIZE` first.

In [ ]:
MODEL_ID    = "google/byt5-small"
DEVICE      = "cuda"    # "cuda" | "mps" | "cpu"
AMP         = True      # bf16 mixed precision (autocast) — keep True on A100/Ampere
WINDOW_SIZE = 6
RANKS       = 32
MAX_SAMPLES = 50000     # A100 trains fast; raise for a better model
MAX_LEN     = 1024      # bytes per example (A100 can also do 2048)
BATCH_SIZE  = 16        # A100-40GB: ~16 (CP/HMM) / 32 (FF) | A100-80GB: 32-64 | T4: 2-4
AMP_FLAG    = "--amp" if AMP else ""
# cheat stays OFF (no --cheat anywhere).

## 4. Phase 0 + 1 + 2 — backbone SFT, FF warm-up, FF joint
Trains the shared backbone + the FF draft and produces the **verifier** backbone
(`byt5_standard_lora_phase0`). First run downloads & tokenizes Tülu-3 (cached to Drive).

In [ ]:
cmd = (f"cd src && python training.py --model_id {MODEL_ID} --device {DEVICE} {AMP_FLAG} --head ff "
       f"--window_size {WINDOW_SIZE} --ranks {RANKS} --max_samples {MAX_SAMPLES} "
       f"--max_len {MAX_LEN} --batch_size {BATCH_SIZE} "
       f"--skip_phase_0 false --skip_phase_1 false --skip_phase_2 false")
print(cmd)
!{cmd}

## 5. Phase 2 — CP head
Reuses the Phase-0 backbone, transfers the Phase-1 FF emissions into the CP circuit, joint-trains it.

In [ ]:
cmd = (f"cd src && python training.py --model_id {MODEL_ID} --device {DEVICE} {AMP_FLAG} --head cp "
       f"--window_size {WINDOW_SIZE} --ranks {RANKS} --max_samples {MAX_SAMPLES} "
       f"--max_len {MAX_LEN} --batch_size {BATCH_SIZE} "
       f"--skip_phase_0 true --skip_phase_1 true --skip_phase_2 false")
print(cmd)
!{cmd}

## 6. (optional) Phase 2 — HMM head

In [ ]:
cmd = (f"cd src && python training.py --model_id {MODEL_ID} --device {DEVICE} {AMP_FLAG} --head hmm "
       f"--window_size {WINDOW_SIZE} --ranks {RANKS} --max_samples {MAX_SAMPLES} "
       f"--max_len {MAX_LEN} --batch_size {BATCH_SIZE} "
       f"--skip_phase_0 true --skip_phase_1 true --skip_phase_2 false")
print(cmd)
!{cmd}

## 6b. (optional) Phase 2 — BTree head

Same recipe as CP/HMM: phase-2 only, so the BTree is initialised from the trained FF head
(emissions copied from the FF leaves, every sum gate zeroed -> uniform mixture, so the tree
starts out identical to Fully-Factorised) and then trained jointly. Needs Phase 1's FF
checkpoint present in `saved_models/`.

In [ ]:
cmd = (f"cd src && python training.py --model_id {MODEL_ID} --device {DEVICE} {AMP_FLAG} --head btree "
       f"--window_size {WINDOW_SIZE} --ranks {RANKS} --max_samples {MAX_SAMPLES} "
       f"--max_len {MAX_LEN} --batch_size {BATCH_SIZE} "
       f"--skip_phase_0 true --skip_phase_1 true --skip_phase_2 false")
print(cmd)
!{cmd}

## 7. Sanity check — honest (non-cheat) prediction
With cheat off, the argmax should be a real *prediction* (not a verbatim copy of the ground truth).

In [ ]:
cmd = f"python src/inference.py --device {DEVICE} --head cp --window_size {WINDOW_SIZE} --num_samples 5"
print(cmd)
!{cmd}

## 8. Done
Saved on Drive under `MTPC/saved_models/`:
- `byt5_standard_lora_phase0/` → verifier backbone
- `mtp_head_{ff,cp,hmm}_w6_final.pth` + `lora_{ff,cp,hmm}_w6/` → draft heads + adapters

Then `git pull` locally and run `speculative_inference_testing.R` (already `CHEAT = FALSE`).